In [1]:
import numpy as np
import pygimli as pg
import pygimli.physics.ert as ert
from enum import IntEnum

In [2]:
class RegionMarkers(IntEnum):
    """Markers for geometric regions."""
    BOREHOLE = 1
    CORE = 2
    WORLD = 100

class BoundaryMarkers(IntEnum):
    """Markers for boundary conditions."""
    NEUMANN = -1  # Free surface (top)
    ROBIN = -2    # Mixed/Robin boundary (far-field)
    INTERNAL = 0  # Internal boundary


In [3]:
background_resistivity = 5.0
area_xy=  (20.0, 20.0)
world_xy=(100.0, 100.0)
layer_1d_geometry = np.array([
    [0.0, 5.0, 100.0],
    [5.0, 10.0, 50.0],
    [10.0, 15.0, 200.0]
])

borehole_length = 15.0
borehole_diameter = 0.25

core_area = borehole_diameter / 10.0
borehole_area = core_area / 5.0
world_area = 0.0

abmn_order=(1, 4, 2, 3)
short_spacing=0.5 
long_spacing=2.0 
measuring_spacing=0.05

num_electrodes = int(round(borehole_length / measuring_spacing)) + 1
z_positions = -np.arange(num_electrodes) * measuring_spacing

In [4]:
x_dim, y_dim = area_xy
wx_dim, wy_dim = world_xy

last_layer_boundary = float(layer_1d_geometry[-1, 1])
core_depth = last_layer_boundary + 10.0 * long_spacing

# 1. Create the outer boundary domain (world)
world_depth = max(borehole_length * 2.5, core_depth * 1.5)
world = pg.meshtools.createCube(
    size=[wx_dim, wy_dim, world_depth],
    pos=[0.0, 0.0, -world_depth / 2.0],
    marker=int(RegionMarkers.WORLD),
    area=world_area
)

# Explicitly set the region marker's position to be inside the world
# but outside the core. This is crucial for correct region identification.
for m in world.regionMarkers():
    m.setPos([(wx_dim + x_dim) / 4.0, 0.0, -world_depth / 4.0])

# 2. Create the inner core block for fine resolution
core = pg.meshtools.createCube(
    size=[x_dim, y_dim, core_depth],
    pos=[0.0, 0.0, -core_depth / 2.0],
    marker=int(RegionMarkers.CORE),
    area=core_area
)

# Explicitly set the region marker's position to be inside the core
# but outside the borehole.
for m in core.regionMarkers():
    m.setPos([(x_dim / 2.0 + borehole_diameter / 2.0) / 2.0, 0.0, -core_depth / 4.0])

# 3. Create the borehole cylinder
borehole = pg.meshtools.createCylinder(
    radius=borehole_diameter / 2.0,
    height=borehole_length,
    pos=[0.0, 0.0, -borehole_length / 2.0],
    marker=int(RegionMarkers.BOREHOLE),
    area=borehole_area
)

for m in borehole.regionMarkers():
    # The default center position is fine for the innermost geometry,
    # but we can set it explicitly for clarity.
    m.setPos([0.0, 0.0, -borehole_length / 4.0])

geom = world + core + borehole

# In PyGIMLi, combining geometries with '+' can discard the area constraints 
# on the region markers. We re-apply them directly to the final PLC markers.
for m in geom.regionMarkers():
    if m.marker() == RegionMarkers.WORLD:
        m.setArea(world_area)
    elif m.marker() == RegionMarkers.CORE:
        m.setArea(core_area)
    elif m.marker() == RegionMarkers.BOREHOLE:
        m.setArea(borehole_area)




############################### WHY THIS SECTION IS NEEDED #####################################
# Set strict boundary conditions for ERT and avoid conflicts with region markers.
# createCube assigns boundary markers 1-6 which conflict with our region markers 1, 2, 3...
for bound in geom.boundaries():
    center = bound.center()
    # Top surface must be -1 (Free surface / Neumann)
    if np.isclose(center[2], 0.0, atol=1e-3):
        bound.setMarker(int(BoundaryMarkers.NEUMANN))
    # Outer walls and bottom must be -2 (Mixed / Robin boundary)
    elif (np.isclose(abs(center[0]), wx_dim / 2.0, atol=1e-3) or
          np.isclose(abs(center[1]), wy_dim / 2.0, atol=1e-3) or
          np.isclose(center[2], -world_depth, atol=1e-3)):
        bound.setMarker(int(BoundaryMarkers.ROBIN))
    else:
        # Internal boundaries should be 0 so they are not treated as Dirichlet BCs
        bound.setMarker(int(BoundaryMarkers.INTERNAL))


# Inject the electrode nodes structurally into the PLC before meshing.
# Connect them with edges to form a constrained 1D line inside the borehole.
# This mathematically forces TetGen to preserve every single electrode exactly as a mesh node.
nodes = []
for z in z_positions:
    node = geom.createNode([0.0, 0.0, z])
    node.setMarker(99)
    nodes.append(node)

for i in range(len(nodes) - 1):
    geom.createEdge(nodes[i], nodes[i+1])

# Create the ERT DataContainer
data = pg.DataContainerERT()
data.setSensorPositions([[0.0, 0.0, z] for z in z_positions])
    
# Map A, B, M, N to their relative index positions (0-based)
a_pos, b_pos, m_pos, n_pos = [idx - 1 for idx in abmn_order]

# Use only the short and long spacings
a_spacings = [short_spacing, long_spacing]

a_idx, b_idx, m_idx, n_idx = [], [], [], []

for a in a_spacings:
    # Number of index steps between adjacent logical array positions
    a_steps = int(round(a / measuring_spacing))
    if a_steps == 0:
        continue
        
    max_rel_idx = max(a_pos, b_pos, m_pos, n_pos) * a_steps
    
    # Move the array from top to bottom
    for start_idx in range(len(z_positions) - max_rel_idx):
        a_idx.append(int(start_idx + a_pos * a_steps))
        b_idx.append(int(start_idx + b_pos * a_steps))
        m_idx.append(int(start_idx + m_pos * a_steps))
        n_idx.append(int(start_idx + n_pos * a_steps))
        
# Register standard arrays for pyGIMLi
data.resize(len(a_idx))
data.set("a", a_idx)
data.set("b", b_idx)
data.set("m", m_idx)
data.set("n", n_idx)
data.set("valid", np.ones(len(a_idx), dtype=int))







# Generate the mesh (omitting global area constraint to respect detailed region area targets)
mesh = pg.meshtools.createMesh(geom, quality=34.2)

# Paint the 1D layers onto both the inner core and outer world cells
for cell in mesh.cells():
    z_center = cell.center()[2]
    if cell.marker() in [RegionMarkers.CORE, RegionMarkers.WORLD]:  # Update both inner core and outer world cells
        for i, row in enumerate(layer_1d_geometry):
            depth_top = float(row[0])
            depth_bottom = float(row[1])
            if i == len(layer_1d_geometry) - 1:
                depth_bottom = 99999.0  # Extend the last layer downwards to the bottom of the world domain
            
            z_top = -depth_top
            z_bottom = -depth_bottom
            
            if z_bottom < z_center <= z_top:
                cell.setMarker(i + 2)
                break

# Create a rhomap defining marker-to-resistivity relationships
# Marker 1 (Borehole) defaults to background_resistivity.
rhomap = [[1, background_resistivity]]

for i, row in enumerate(layer_1d_geometry):
    res_value = float(row[2]) if len(row) > 2 else np.nan
    rhomap.append([i + 2, res_value])

# Store the rhomap on the instance for future ERT simulation
rhomap = rhomap

# Map the rhomap to the mesh cells to create the Resistivity array for VTK export
rhomap_dict = dict(rhomap)
mesh["Resistivity"] = np.array([rhomap_dict.get(cell.marker(), np.nan) for cell in mesh.cells()])



rhomap_dict = dict(rhomap)
res_list = [rhomap_dict.get(cell.marker(), background_resistivity) for cell in mesh.cells()]

# Convert explicitly to pyGIMLi Vector to prevent internal region mapping crashes
res_vec = pg.Vector(res_list)


sim_data = ert.simulate(
    mesh=mesh,
    scheme=data,
    res=res_vec,
    background=background_resistivity,
    noiseLevel=0.0,
    noiseAbs=0.0,
    sr=True  # Use Singularity Removal for accurate primary potential calculation
)


sim_data.save('output_data.dat')

19/06/26 - 12:16:00 - pyGIMLi - INFO - Calculate geometric factors.


1